## Phase 3.5 — Severity Head Training on DAIC-WOZ

This notebook completes Phase 3 by training the **severity regression head** — the one component that was frozen throughout Phase 3 due to missing DAIC-WOZ data access. All other components (MentalRoBERTa backbone, GAT layers, emotion head, MH category head) carry forward their trained weights from earlier phases.

### What this notebook does
- Loads the DAIC-WOZ clinical interview dataset (AVEC 2017 split)
- Extracts participant speech transcripts and maps them to PHQ-8 severity scores
- Unfreezes and trains the severity head jointly with all other layers
- Evaluates PHQ-8 Mean Absolute Error (MAE) against published benchmarks

### Why PHQ-8 and not PHQ-9?
DAIC-WOZ uses PHQ-8 (score range 0–24) rather than PHQ-9. The ninth item (suicidal ideation) was excluded from the original dataset for ethical reasons during data collection. This is standard for this benchmark — all published results on DAIC-WOZ use PHQ-8.

This cell discovers the label CSV files inside the DAIC-WOZ input folder.

In [24]:
import glob

# correct base path
base = "/kaggle/input/datasets"

# find any label/score/phq files
for pattern in ["**/*.csv", "**/*.txt"]:
    for f in glob.glob(f"{base}/{pattern}", recursive=True):
        if any(x in f.lower() for x in ["label", "score", "phq", "train", "test", "dev", "split"]):
            print(f)

/kaggle/input/datasets/saifzaman123445/daicwoz/daicwoz/daicwoz/test_split_Depression_AVEC2017.csv
/kaggle/input/datasets/saifzaman123445/daicwoz/daicwoz/daicwoz/dev_split_Depression_AVEC2017.csv
/kaggle/input/datasets/saifzaman123445/daicwoz/daicwoz/daicwoz/full_test_split.csv
/kaggle/input/datasets/saifzaman123445/daicwoz/daicwoz/daicwoz/train_split_Depression_AVEC2017.csv


### Dataset folder structure

DAIC-WOZ on Kaggle is nested under a user-specific path (`/kaggle/input/datasets/saifzaman123445/...`). This cell confirms the top-level folder layout so the correct base path can be hardcoded for all subsequent file reads.

In [25]:
import os
for item in os.listdir("/kaggle/input/datasets"):
    print(item)
    sub = f"/kaggle/input/datasets/{item}"
    if os.path.isdir(sub):
        for sub2 in os.listdir(sub)[:5]:
            print(f"  {sub2}")

saifzaman123445
  daicwoz


### Label file inspection

The train split contains **107 participants** with 12 columns: `Participant_ID`, `PHQ8_Binary` (0=not depressed, 1=depressed), `PHQ8_Score` (0–24 continuous severity), gender, and the 8 individual PHQ sub-scores (NoInterest, Depressed, Sleep, Tired, Appetite, Failure, Concentrating, Moving).

The split is by **participant ID** — the same participant never appears in both train and validation. This is critical for DAIC-WOZ: if split randomly, early and late sessions from the same participant could leak context between train and val, artificially inflating validation scores.

In [26]:
import pandas as pd

DAIC_BASE = "/kaggle/input/datasets/saifzaman123445/daicwoz/daicwoz/daicwoz"

train_labels = pd.read_csv(f"{DAIC_BASE}/train_split_Depression_AVEC2017.csv")
dev_labels   = pd.read_csv(f"{DAIC_BASE}/dev_split_Depression_AVEC2017.csv")

print(train_labels.shape)
print(train_labels.columns.tolist())
print(train_labels.head())

(107, 12)
['Participant_ID', 'PHQ8_Binary', 'PHQ8_Score', 'Gender', 'PHQ8_NoInterest', 'PHQ8_Depressed', 'PHQ8_Sleep', 'PHQ8_Tired', 'PHQ8_Appetite', 'PHQ8_Failure', 'PHQ8_Concentrating', 'PHQ8_Moving']
   Participant_ID  PHQ8_Binary  PHQ8_Score  Gender  PHQ8_NoInterest  \
0             303            0           0       0                0   
1             304            0           6       0                0   
2             305            0           7       1                0   
3             310            0           4       1                1   
4             312            0           2       1                0   

   PHQ8_Depressed  PHQ8_Sleep  PHQ8_Tired  PHQ8_Appetite  PHQ8_Failure  \
0               0         0.0           0              0             0   
1               1         1.0           2              2             0   
2               1         1.0           2              2             1   
3               1         0.0           0              0             1   


## 1. Imports and Checkpoint Discovery

Standard imports plus dynamic path discovery for Phase 1 and Phase 3 checkpoints. The model needs both:
- **Phase 1** (`mentalroberta_phase1_final`): provides the tokenizer and the pretrained backbone weights
- **Phase 3** (`phase3_best.pt`): provides the trained GAT layers, emotion head, and MH head weights to carry forward

If either path is `None`, the corresponding weights fall back to random initialisation — acceptable for smoke-testing but not for real training.

In [27]:
import os, glob, re, torch, numpy as np, pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, f1_score
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

DAIC_BASE  = "/kaggle/input/datasets/saifzaman123445/daicwoz/daicwoz/daicwoz"
PHASE1_PATH = glob.glob("/kaggle/input/notebooks/**/mentalroberta_phase1_final", recursive=True)
PHASE1_PATH = PHASE1_PATH[0] if PHASE1_PATH else None
PHASE3_PATH = glob.glob("/kaggle/input/notebooks/**/phase3_best.pt", recursive=True)
PHASE3_PATH = PHASE3_PATH[0] if PHASE3_PATH else None

print("Phase 1:", PHASE1_PATH)
print("Phase 3:", PHASE3_PATH)

Device: cuda
Phase 1: /kaggle/input/notebooks/shashwatkashyap12221/phase-1-mental-health-monitoring-system/mentalroberta_phase1_final
Phase 3: /kaggle/input/notebooks/shashwatkashyap12221/phase-3-mental-health-monitoring-system/phase3_best.pt


## 2. Transcript Loading and Data Preparation

Each DAIC-WOZ participant has a `_TRANSCRIPT.csv` file containing timestamped speech from both the virtual interviewer (Ellie) and the participant. Only **participant speech** is kept — interviewer utterances are filtered out before tokenisation.

This is an important design decision. Research has shown that including interviewer speech can cause models to learn shortcuts from the phrasing of questions rather than genuine depression signal from participant responses (Li et al., 2024 — IEEE Trans. Computational Social Systems).

The `build_pairs()` function returns a list of `{text, score, binary, pid}` dicts. Participants with fewer than 50 characters of usable transcript are dropped (these are typically sessions where the participant gave very short or non-verbal responses that didn't transcribe meaningfully).

**Output interpretation:** `Train: X | Val: Y` confirms how many participants successfully loaded transcripts from each split. Some participants may be missing transcript files if the Kaggle dataset upload was incomplete — the code handles this gracefully by skipping them.

In [28]:
train_labels = pd.read_csv(f"{DAIC_BASE}/train_split_Depression_AVEC2017.csv")
dev_labels   = pd.read_csv(f"{DAIC_BASE}/dev_split_Depression_AVEC2017.csv")

def load_transcript(participant_id):
    """Load and concatenate participant transcript text."""
    # transcripts are in folders named by participant ID
    patterns = [
        f"{DAIC_BASE}/{participant_id}/{participant_id}_TRANSCRIPT.csv",
        f"{DAIC_BASE}/{participant_id}_TRANSCRIPT.csv",
    ]
    for p in patterns:
        if os.path.exists(p):
            try:
                df = pd.read_csv(p, sep='\t')
                # keep only participant speech, not interviewer
                if 'speaker' in df.columns and 'value' in df.columns:
                    text = " ".join(df[df['speaker']=='Participant']['value'].dropna().astype(str).tolist())
                elif 'value' in df.columns:
                    text = " ".join(df['value'].dropna().astype(str).tolist())
                else:
                    text = " ".join(df.iloc[:,1].dropna().astype(str).tolist())
                return text.strip()
            except:
                pass
    return None

# build text + label pairs, split by participant ID (never random split)
def build_pairs(label_df):
    pairs = []
    for _, row in label_df.iterrows():
        pid = int(row['Participant_ID'])
        text = load_transcript(pid)
        if text and len(text) > 50:
            score = float(row['PHQ8_Score'])
            pairs.append({"text": text, "score": score,
                          "binary": int(row['PHQ8_Binary']), "pid": pid})
    return pairs

train_pairs = build_pairs(train_labels)
val_pairs   = build_pairs(dev_labels)
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)}")
print(f"Score range — Train: {min(p['score'] for p in train_pairs):.0f}–{max(p['score'] for p in train_pairs):.0f}")

Train: 107 | Val: 34
Score range — Train: 0–20


## 3. Tokenizer and Dataset Class

The MentalRoBERTa tokenizer from Phase 1 is reused. Transcripts are truncated to **256 tokens** — shorter than Phase 1's 128 because interview transcripts are long-form text where context further into the conversation matters for severity estimation.

The `DAICDataset` returns five items per sample:
- `input_ids`, `attention_mask` — standard tokenizer outputs
- `severity` — PHQ-8 score **normalised to 0–1** by dividing by 24 (the maximum possible score). This is required before computing MSE loss — if the raw 0–24 range were used, the severity loss would be ~576× larger in scale than the cross-entropy MH loss and would completely dominate training.
- `binary` — 0/1 depression label for the MH head
- `raw_score` — original PHQ-8 score kept separately for MAE computation at evaluation time (we multiply predictions back by 24 to get interpretable PHQ-8 scale numbers)

In [29]:
tokenizer = AutoTokenizer.from_pretrained(PHASE1_PATH)

class DAICDataset(Dataset):
    def __init__(self, pairs, max_len=256):
        self.pairs  = pairs
        self.max_len = max_len

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        enc = tokenizer(p["text"], truncation=True, padding="max_length",
                        max_length=self.max_len, return_tensors="pt")
        # normalize PHQ8 score to 0-1 range (max=24)
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "severity":       torch.tensor(p["score"] / 24.0, dtype=torch.float),
            "binary":         torch.tensor(p["binary"], dtype=torch.long),
            "raw_score":      torch.tensor(p["score"], dtype=torch.float),
        }

train_ds = DAICDataset(train_pairs)
val_ds   = DAICDataset(val_pairs)
train_loader = DataLoader(train_ds, batch_size=8,  shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 14 | Val batches: 3


## 4. Install torch_geometric

`torch_geometric` must be installed fresh in this notebook since it's a new Kaggle session. This is the library that provides the `GATConv` layers used in the Phase 2 and Phase 3 architecture.

In [30]:
!pip install -q torch_geometric
import torch
print(torch.__version__)

2.10.0+cu128


## 5. Phase 3.5 Model Architecture

The model is structurally identical to Phase 3, with one critical difference: **the severity head is now fully trainable** (no `requires_grad = False`).

The architecture:
- **Backbone**: MentalRoBERTa (768-dim token embeddings)
- **GAT Layer 1**: 768 → 4 heads × 256 = 1024-dim (matches Phase 2 exactly)
- **GAT Layer 2**: 1024 → 2 heads → 256-dim (matches Phase 2 exactly)
- **Emotion head**: 256 → 128 → 28 outputs (multi-label sigmoid, carries Phase 3 weights)
- **MH head**: 256 → 64 → 2 outputs (binary classification, carries Phase 3 weights)
- **Severity head**: 256 → 128 → 64 → 1 output with Sigmoid (newly trainable, trains to 0–1 range matching normalised PHQ-8)

The `_edge_cache` dictionary pre-builds token-to-token graph edges for each unique (batch_size, seq_len) combination so they are not recomputed on every forward pass — a significant speed improvement when training on long transcripts.

In [31]:
from torch_geometric.nn import GATConv

class Phase35Model(nn.Module):
    def __init__(self, backbone_path, num_emotions=28):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_path)
        self.gat1 = GATConv(768,  256, heads=4, concat=True)
        self.gat2 = GATConv(1024, 256, heads=2, concat=False)

        self.emotion_head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, num_emotions)
        )
        self.mh_head = nn.Sequential(
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 2)
        )
        # UNFROZEN severity head — trained on DAIC-WOZ
        self.severity_head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64),  nn.ReLU(),
            nn.Linear(64, 1),    nn.Sigmoid()
        )
        self._edge_cache = {}

    def _get_edges(self, B, S):
        key = (B, S)
        if key not in self._edge_cache:
            src, dst = [], []
            for b in range(B):
                offset = b * S
                s = torch.arange(offset, offset + S - 1)
                d = torch.arange(offset + 1, offset + S)
                src.append(s); dst.append(d)
            self._edge_cache[key] = torch.stack(
                [torch.cat(src), torch.cat(dst)]
            )
        return self._edge_cache[key].to(device)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids,
                            attention_mask=attention_mask)
        B, S, H = out.last_hidden_state.shape
        x = out.last_hidden_state.reshape(B * S, H)
        edge_index = self._get_edges(B, S)
        x = torch.relu(self.gat1(x, edge_index))
        x = torch.relu(self.gat2(x, edge_index))
        cls_idx = torch.arange(B, device=x.device) * S
        vec = x[cls_idx]
        return (self.emotion_head(vec),
                self.mh_head(vec),
                self.severity_head(vec).squeeze(-1))

model = Phase35Model(PHASE1_PATH).to(device)
print("Phase 3.5 model ready — severity head NOW trainable")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: /kaggle/input/notebooks/shashwatkashyap12221/phase-1-mental-health-monitoring-system/mentalroberta_phase1_final
Key                        | Status     | 
---------------------------+------------+-
classifier.out_proj.weight | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Phase 3.5 model ready — severity head NOW trainable


## 6. Transfer Weights from Phase 3

All matching weights from the Phase 3 checkpoint are loaded into the model. This transfers:
- The trained GAT layers (Phase 2 knowledge about clinical grammatical relationships)
- The trained emotion head (Phase 3 knowledge from GoEmotions)
- The trained MH category head (Phase 3 binary depression signal)

The severity head loads randomly since it was frozen and untrained in Phase 3 — this is expected and correct. The `{matched}/{total}` count confirms how many parameter tensors matched by name and shape.

In [32]:
if PHASE3_PATH:
    ckpt = torch.load(PHASE3_PATH, map_location=device)
    model_dict = model.state_dict()
    matched = {k: v for k, v in ckpt.items()
               if k in model_dict and model_dict[k].shape == v.shape}
    model_dict.update(matched)
    model.load_state_dict(model_dict)
    print(f"Loaded {len(matched)}/{len(ckpt)} layers from Phase 3")
else:
    print("No Phase 3 checkpoint — starting from Phase 1 only")

Loaded 215/219 layers from Phase 3


## 7. Loss Functions and Differential Learning Rates

Three loss functions, two used per batch:
- `MSELoss` for severity regression (normalised 0–1 predictions vs normalised labels)
- `CrossEntropyLoss` for binary MH classification
- `BCEWithLogitsLoss` defined but not used in this notebook (emotion head not trained here since DAIC-WOZ has no emotion labels)

**Combined loss**: `0.6 × MSE_severity + 0.4 × CE_mh` — severity gets higher weight since it's the primary new objective of this phase.

**Differential learning rates** are critical here:
- Backbone: `1e-5` — very low to prevent destroying Phase 1 and Phase 3 knowledge
- GAT layers and heads: `3e-5` — moderate, allows gentle adaptation
- Severity head: `1e-4` — highest rate since it's training from a random initialisation and needs to learn faster than the already-trained components

Using `weight_decay=0.01` (L2 regularisation) is important given the small dataset size (~107 participants) to prevent the severity head from overfitting.

In [33]:
mse_loss = nn.MSELoss()
bce      = nn.BCEWithLogitsLoss()
ce       = nn.CrossEntropyLoss()

# severity head gets highest lr since it's training from scratch
# backbone stays low to preserve Phase 1+3 learning
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(),      "lr": 1e-5},
    {"params": model.gat1.parameters(),          "lr": 3e-5},
    {"params": model.gat2.parameters(),          "lr": 3e-5},
    {"params": model.emotion_head.parameters(),  "lr": 3e-5},
    {"params": model.mh_head.parameters(),       "lr": 3e-5},
    {"params": model.severity_head.parameters(), "lr": 1e-4},
], weight_decay=0.01)

print("Optimizer ready — severity head lr=1e-4, backbone lr=1e-5")

Optimizer ready — severity head lr=1e-4, backbone lr=1e-5


## 8. Training Loop — 100 Epochs with Early Stopping

The training loop runs for up to 100 epochs with **patience=10 early stopping** on validation MAE. If MAE does not improve for 10 consecutive epochs, training terminates automatically.

Each epoch prints one line:
- `Loss`: average training loss across all batches
- `MAE`: Mean Absolute Error on the validation set in PHQ-8 units (0–24 scale)
- `MH F1`: macro F1 for binary depression classification on the validation set
- `✓ best` when a new best MAE is achieved and the checkpoint is saved

**Why MAE and not MSE for evaluation?** MAE is directly interpretable — an MAE of 4.0 means predictions are off by 4 PHQ-8 points on average. MSE penalises large errors disproportionately and is harder to interpret clinically. Published DAIC-WOZ results always report MAE, making this the correct metric for benchmark comparison.

**Published benchmark**: the best text-only result on DAIC-WOZ is MAE ≈ 2.85 (LLM-based approach using PHQ-8 indicator extraction). Our system is a smaller model using interview transcripts without audio features, so a higher MAE is expected.

In [34]:
EPOCHS   = 100
best_mae = float("inf")
patience = 10
no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        ids      = batch["input_ids"].to(device)
        mask     = batch["attention_mask"].to(device)
        severity = batch["severity"].to(device)
        binary   = batch["binary"].to(device)

        optimizer.zero_grad()
        emo_logits, mh_logits, sev_pred = model(ids, mask)

        loss_sev = mse_loss(sev_pred, severity)
        loss_mh  = ce(mh_logits, binary)
        loss = 0.6 * loss_sev + 0.4 * loss_mh
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # validation
    model.eval()
    all_sev_pred, all_sev_true = [], []
    all_mh_pred,  all_mh_true  = [], []

    with torch.no_grad():
        for batch in val_loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            _, mh_logits, sev_pred = model(ids, mask)
            all_sev_pred.extend((sev_pred.cpu() * 24).numpy())
            all_sev_true.extend(batch["raw_score"].numpy())
            all_mh_pred.extend(mh_logits.argmax(-1).cpu().numpy())
            all_mh_true.extend(batch["binary"].numpy())

    mae   = mean_absolute_error(all_sev_true, all_sev_pred)
    mh_f1 = f1_score(all_mh_true, all_mh_pred, average="macro", zero_division=0)
    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | MAE: {mae:.2f} | MH F1: {mh_f1:.4f}", end="")

    if mae < best_mae:
        best_mae = mae
        no_improve = 0
        torch.save(model.state_dict(), "/kaggle/working/phase35_best.pt")
        print(" ✓ best")
    else:
        no_improve += 1
        print(f" (no improvement {no_improve}/{patience})")

    if no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch+1} — no MAE improvement for {patience} epochs.")
        break

print(f"\nTraining complete. Best MAE: {best_mae:.2f}")

Epoch   1/100 | Loss: 2.5827 | MAE: 7.03 | MH F1: 0.2609 ✓ best
Epoch   2/100 | Loss: 0.4066 | MAE: 7.10 | MH F1: 0.5228 (no improvement 1/10)
Epoch   3/100 | Loss: 0.3343 | MAE: 6.94 | MH F1: 0.4603 ✓ best
Epoch   4/100 | Loss: 0.3222 | MAE: 6.60 | MH F1: 0.3929 ✓ best
Epoch   5/100 | Loss: 0.3039 | MAE: 6.07 | MH F1: 0.3929 ✓ best
Epoch   6/100 | Loss: 0.2875 | MAE: 5.68 | MH F1: 0.3929 ✓ best
Epoch   7/100 | Loss: 0.2745 | MAE: 5.45 | MH F1: 0.3929 ✓ best
Epoch   8/100 | Loss: 0.2729 | MAE: 5.40 | MH F1: 0.3929 ✓ best
Epoch   9/100 | Loss: 0.2686 | MAE: 5.38 | MH F1: 0.3929 ✓ best
Epoch  10/100 | Loss: 0.2592 | MAE: 5.31 | MH F1: 0.3929 ✓ best
Epoch  11/100 | Loss: 0.2526 | MAE: 5.38 | MH F1: 0.5296 (no improvement 1/10)
Epoch  12/100 | Loss: 0.2476 | MAE: 5.20 | MH F1: 0.5503 ✓ best
Epoch  13/100 | Loss: 0.2352 | MAE: 5.08 | MH F1: 0.6458 ✓ best
Epoch  14/100 | Loss: 0.1989 | MAE: 4.95 | MH F1: 0.6713 ✓ best
Epoch  15/100 | Loss: 0.1821 | MAE: 4.80 | MH F1: 0.6762 ✓ best
Epoch  16/

## 9. Final Evaluation on Validation Set

The best checkpoint (lowest validation MAE across all epochs) is reloaded and evaluated. Severity predictions are multiplied back by 24 to return to the interpretable PHQ-8 scale before computing MAE.

**Result interpretation:**
- **PHQ-8 MAE of 4.44** means the model's severity estimates are off by ~4.6 points on the 0–24 scale. Given the PHQ-8 clinical thresholds (0–4 none, 5–9 mild, 10–14 moderate, 15–19 moderately severe, 20–24 severe), an error of 4.44 is roughly one clinical severity band — the model identifies the right region but not always the exact sub-score.
- **Depression F1-macro of 0.7509** is strong for binary classification and consistent with the MH head's performance from Phase 3.
- The gap from the published 2.85 MAE benchmark is partly explained by dataset size (107 training participants vs larger LLM studies), absence of audio features (prosody and speech rate are strong depression indicators), and the fact that our model is a general-purpose multi-task system rather than a DAIC-WOZ-specific model.

The per-participant predictions show the model is tracking the right direction — participants with low PHQ-8 scores (0, 4) tend to get low predictions, and participants with high scores (19, 23) get higher predictions, even if not perfectly calibrated.

Both `phase35_best.pt` and `phase35_final.pt` are saved to `/kaggle/working/` along with the tokenizer. These serve as the final classifier checkpoint for Phase 4 (Groq LLM integration).

In [35]:
model.load_state_dict(torch.load("/kaggle/working/phase35_best.pt"))
model.eval()

all_sev_pred, all_sev_true = [], []
all_mh_pred,  all_mh_true  = [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Final eval"):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        _, mh_logits, sev_pred = model(ids, mask)
        all_sev_pred.extend((sev_pred.cpu() * 24).numpy())
        all_sev_true.extend(batch["raw_score"].numpy())
        all_mh_pred.extend(mh_logits.argmax(-1).cpu().numpy())
        all_mh_true.extend(batch["binary"].numpy())

mae = mean_absolute_error(all_sev_true, all_sev_pred)
mh_f1 = f1_score(all_mh_true, all_mh_pred, average="macro", zero_division=0)

print("=== PHASE 3.5 — SEVERITY HEAD RESULTS ===")
print(f"PHQ-8 MAE:          {mae:.2f}  (published best text-only: ~2.85)")
print(f"Depression F1-macro: {mh_f1:.4f}")
print(f"\nPer-participant predictions vs ground truth:")
for pred, true in zip(all_sev_pred[:10], all_sev_true[:10]):
    print(f"  Predicted: {pred:.1f}  |  Actual: {true:.0f}")

# save final
torch.save(model.state_dict(), "/kaggle/working/phase35_final.pt")
tokenizer.save_pretrained("/kaggle/working/phase35_tokenizer")
print("\nPhase 3.5 complete. All checkpoints saved.")

Final eval: 100%|██████████| 3/3 [00:00<00:00,  4.78it/s]


=== PHASE 3.5 — SEVERITY HEAD RESULTS ===
PHQ-8 MAE:          4.44  (published best text-only: ~2.85)
Depression F1-macro: 0.7509

Per-participant predictions vs ground truth:
  Predicted: 2.2  |  Actual: 4
  Predicted: 14.3  |  Actual: 4
  Predicted: 11.6  |  Actual: 8
  Predicted: 9.6  |  Actual: 12
  Predicted: 12.8  |  Actual: 23
  Predicted: 15.1  |  Actual: 19
  Predicted: 12.6  |  Actual: 16
  Predicted: 9.8  |  Actual: 16
  Predicted: 2.5  |  Actual: 0
  Predicted: 14.4  |  Actual: 17

Phase 3.5 complete. All checkpoints saved.
